# import data

In [26]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path="data.json",
    jq_schema=".[]",      # ambil setiap item dalam array
    text_content=False   # biar otomatis string
)

docs = loader.load()
print(f"Jumlah dokumen: {len(docs)}")
print(docs[0].page_content)


Jumlah dokumen: 400
{"question": "Apa nama ilmiah tanaman tebu yang disebutkan dalam dokumen?", "answer": "Saccharum officinarum L.", "answer_type": "explicit", "evidence": "Tanaman tebu (Saccharum officinarum L.) merupakan sejenis rerumputan yang digolongkan dalam famili Graminae dan dikenal sebagai penghasil gula."}


In [27]:
print(docs[0].page_content[:500])
print(docs[0].metadata)


{"question": "Apa nama ilmiah tanaman tebu yang disebutkan dalam dokumen?", "answer": "Saccharum officinarum L.", "answer_type": "explicit", "evidence": "Tanaman tebu (Saccharum officinarum L.) merupakan sejenis rerumputan yang digolongkan dalam famili Graminae dan dikenal sebagai penghasil gula."}
{'source': '/Users/muhammadzuamaalamin/Documents/dataprep/analisisdata/cek2/data.json', 'seq_num': 1}


In [28]:
for i in range(len(docs)):
    print(docs[i].page_content)

{"question": "Apa nama ilmiah tanaman tebu yang disebutkan dalam dokumen?", "answer": "Saccharum officinarum L.", "answer_type": "explicit", "evidence": "Tanaman tebu (Saccharum officinarum L.) merupakan sejenis rerumputan yang digolongkan dalam famili Graminae dan dikenal sebagai penghasil gula."}
{"question": "Dalam keluarga tumbuhan apa tanaman tebu diklasifikasikan?", "answer": "Famili Graminae.", "answer_type": "explicit", "evidence": "Tanaman tebu (Saccharum officinarum L.) merupakan sejenis rerumputan yang digolongkan dalam famili Graminae dan dikenal sebagai penghasil gula."}
{"question": "Mengapa gula dianggap penting dalam kehidupan masyarakat menurut dokumen?", "answer": "Karena gula merupakan salah satu kebutuhan pokok dan sumber kalori yang relatif murah.", "answer_type": "explicit", "evidence": "Gula merupakan salah satu kebutuhan pokok dan sebagai sumber kalori yang relatif murah."}
{"question": "Berapa jumlah pekebun tebu yang menggantungkan pendapatan pada industri gul

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

texts = text_splitter.split_documents(docs)
texts

[Document(metadata={'source': '/Users/muhammadzuamaalamin/Documents/dataprep/analisisdata/cek2/data.json', 'seq_num': 1}, page_content='{"question": "Apa nama ilmiah tanaman tebu yang disebutkan dalam dokumen?", "answer": "Saccharum officinarum L.", "answer_type": "explicit", "evidence": "Tanaman tebu (Saccharum officinarum L.) merupakan sejenis rerumputan yang digolongkan dalam famili Graminae dan dikenal sebagai penghasil gula."}'),
 Document(metadata={'source': '/Users/muhammadzuamaalamin/Documents/dataprep/analisisdata/cek2/data.json', 'seq_num': 2}, page_content='{"question": "Dalam keluarga tumbuhan apa tanaman tebu diklasifikasikan?", "answer": "Famili Graminae.", "answer_type": "explicit", "evidence": "Tanaman tebu (Saccharum officinarum L.) merupakan sejenis rerumputan yang digolongkan dalam famili Graminae dan dikenal sebagai penghasil gula."}'),
 Document(metadata={'source': '/Users/muhammadzuamaalamin/Documents/dataprep/analisisdata/cek2/data.json', 'seq_num': 3}, page_cont

In [30]:
texts[0].page_content


'{"question": "Apa nama ilmiah tanaman tebu yang disebutkan dalam dokumen?", "answer": "Saccharum officinarum L.", "answer_type": "explicit", "evidence": "Tanaman tebu (Saccharum officinarum L.) merupakan sejenis rerumputan yang digolongkan dalam famili Graminae dan dikenal sebagai penghasil gula."}'

# simpan vector

In [31]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os

embeddings = HuggingFaceEmbeddings(
    model_name="/Users/muhammadzuamaalamin/Documents/fintunellm/model/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

db_path = "faiss_indexallmini"

if not os.path.exists(db_path):
    vectorstore = FAISS.from_documents(texts, embeddings)
    vectorstore.save_local(db_path)
else:
    vectorstore = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )


In [32]:
query = "Tanaman tebu adalah"

results = vectorstore.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i} | score={score:.4f}")
    print(f"Halaman: {doc.metadata.get('page')}")
    print(doc.page_content[:400])



Result 1 | score=0.6767
Halaman: None
. Lahan Kering adalah lahan yang mengandalkan air hujan untuk memenuhi kebutuhan tanaman."}

Result 2 | score=0.7601
Halaman: None
untuk tanaman tebu."], "metode_inspeksi": ["Monitoring hama dan penyakit serta teknik pengendalian di lapangan.", "Mengecek label dan penanganan kemasan bekas pestisida yang digunakan petani."]}, {"tahapan": "Panen (Tebang, Muat, dan Angkut)", "persyaratan_budidaya_tebu_giling_yang_baik": ["Panen tebu dilaksanakan sesuai dengan pola giling pabrik gula.", "Pelaksanaan Tebang, Muat dan Angkut (TMA) 

Result 3 | score=0.7708
Halaman: None
{"question": "Apa yang dimaksud dengan Ratoon dalam konteks budidaya tebu?", "answer": "Tanaman tebu yang tumbuh dari tunas tanaman sebelumnya setelah ditebang.", "answer_type": "explicit", "evidence": "12. Ratoon (Keprasan) adalah tanaman tebu yang tumbuh dari tunas tanaman sebelumnya (setelah ditebang)."}

Result 4 | score=0.7971
Halaman: None
{"question": "Apa yang dimaksud dengan Tur

In [33]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

bm25 = BM25Retriever.from_documents(texts)
bm25.k = 5

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

In [34]:
query = "Tanaman tebu adalah"

docs = hybrid_retriever.get_relevant_documents(query)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i} | score={score:.4f}")
    print(f"Halaman: {doc.metadata.get('page')}")
    print(doc.page_content[:400])



Result 1 | score=0.6767
Halaman: None
. Lahan Kering adalah lahan yang mengandalkan air hujan untuk memenuhi kebutuhan tanaman."}

Result 2 | score=0.7601
Halaman: None
untuk tanaman tebu."], "metode_inspeksi": ["Monitoring hama dan penyakit serta teknik pengendalian di lapangan.", "Mengecek label dan penanganan kemasan bekas pestisida yang digunakan petani."]}, {"tahapan": "Panen (Tebang, Muat, dan Angkut)", "persyaratan_budidaya_tebu_giling_yang_baik": ["Panen tebu dilaksanakan sesuai dengan pola giling pabrik gula.", "Pelaksanaan Tebang, Muat dan Angkut (TMA) 

Result 3 | score=0.7708
Halaman: None
{"question": "Apa yang dimaksud dengan Ratoon dalam konteks budidaya tebu?", "answer": "Tanaman tebu yang tumbuh dari tunas tanaman sebelumnya setelah ditebang.", "answer_type": "explicit", "evidence": "12. Ratoon (Keprasan) adalah tanaman tebu yang tumbuh dari tunas tanaman sebelumnya (setelah ditebang)."}

Result 4 | score=0.7971
Halaman: None
{"question": "Apa yang dimaksud dengan Tur

# model

In [35]:
from langchain.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("""
Kamu adalah asisten ahli pertanian.

Jawablah pertanyaan berikut HANYA berdasarkan konteks dokumen.

Pertanyaan:
{question}

Konteks:
{context}

Jika jawaban tidak ditemukan dalam dokumen, katakan:
"Maaf, informasi tersebut tidak dijelaskan secara spesifik dalam Peraturan Menteri Pertanian Nomor 53 Tahun 2015."
""")


# model 1b

In [36]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Berapa ukuran lebar atas got keliling dalam Sistem Reynoso"}
)

print(response["result"])


Lebar atas got keliling adalah 70 cm.


In [37]:
def run_qa(chain, question):
    response = chain.invoke({"query": question})
    return response["result"]
import json

with open("test1.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

questions = [item["question"] for item in test_data]
references = [item["answer"] for item in test_data]


In [38]:
questions

['Apa yang dimaksud dengan tanaman tebu?',
 'Apa kepanjangan dari MBS dalam konteks tebu layak giling?',
 'Berapa kisaran curah hujan yang sesuai untuk budidaya tebu?',
 'Apa saja dua pola tanam tebu yang disebutkan dalam pedoman?',
 'Apa fungsi utama kegiatan klentek pada tanaman tebu?',
 'Berapa minimal kebutuhan benih bagal mata 2-3 per hektar?',
 'Apa yang dimaksud dengan Plant Cane (PC)?',
 'Berapa minimal umur tebu agar layak giling?',
 'Alat apa yang digunakan untuk mengukur kedalaman permukaan air tanah?',
 'Penyakit apa yang disebabkan oleh bakteri Leaf sonia xyli subsp.xyli?',
 'Apa yang dimaksud dengan varietas tebu unggul?',
 'Sebutkan dua bentuk benih tebu yang dapat digunakan.',
 'Apa nama sistem pengolahan tanah pada lahan sawah yang dikerjakan secara manual dengan pembuatan saluran air?',
 'Apa fungsi pekerjaan gulud pada budidaya tebu?',
 'Hama apakah yang gejala serangannya ditandai dengan cambuk hitam pada pucuk tanaman?',
 'Berapa batas toleransi kadar kotoran pada 

In [39]:
references

['Tanaman tebu adalah jenis tanaman semusim yang mengandung sukrosa atau yang mengandung kadar gula dan dibudidayakan untuk bahan baku pabrik gula.',
 'Masak, Bersih, dan Segar.',
 'Curah hujan antara 1.000–2.000 milimeter per tahun dengan sekurang-kurangnya 3 bulan kering.',
 'Pola A/I (lahan berpengairan, tanam April-Agustus) dan Pola B/II (lahan tadah hujan, tanam September-November).',
 'Untuk mengelupas daun tebu yang telah kering atau kuning, melancarkan sirkulasi udara dan cahaya, serta mengurangi kelembaban.',
 'Minimal 60.000 mata per hektar.',
 'Tanaman tebu pertama yang berasal dari lahan bukan bekas tebu, menggunakan benih unggul dan bersertifikat.',
 'Minimal 11 (sebelas) bulan.',
 'Pipa Piezometer.',
 'Penyakit Pembuluh Ratoon Stunting Disease (RSD).',
 'Varietas tebu yang menunjukkan adaptasi dan produktivitas yang tinggi, serta memiliki keunggulan-keunggulan tertentu baik dari aspek keragaan tanaman maupun parameter pabrikasi.',
 'Benih berupa setek batang/bagal mata 2 

In [40]:
from tqdm.auto import tqdm
import json

results = []      # simpan detail per QA
predictions = []  # hanya jawaban (untuk evaluasi)

for q in tqdm(questions, desc="Running QA Inference", unit="question"):
    try:
        pred = run_qa(qa_chain, q)
    except Exception as e:
        print(f"Error on question: {q}")
        print(e)
        pred = ""

    predictions.append(pred)

    results.append({
        "question": q,
        "prediction": pred
    })



Running QA Inference:   0%|          | 0/20 [00:00<?, ?question/s]

In [41]:
output_data = []

for q, pred, ref in zip(questions, predictions, references):
    output_data.append({
        "question": q,
        "ground_truth": ref,
        "prediction": pred
    })

with open("qa_predictions_gemma3:4btest1newallmini.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

In [42]:
predictions 

['Maaf, informasi tersebut tidak dijelaskan secara spesifik dalam dokumen.',
 'MBS adalah singkatan dari Masak, Bersih, Segar, yang merupakan kriteria dalam pelaksanaan tebang, muat, dan angkut tebu.',
 'Curah hujan antara 1.000–2.000 milimeter per tahun dengan sekurang-kurangnya 3 bulan kering.',
 'Dua sistem penanaman tebu yang disebutkan dalam pedoman adalah sistem manual dan sistem mekanis.',
 'Klentek dilakukan tiga kali atau dapat menyesuaikan kondisi tanaman.',
 'Minimal 60.000 mata per hektar.',
 'Maaf, informasi tersebut tidak dijelaskan secara spesifik dalam Peraturan Menteri Pertanian Nomor 53 Tahun 2015.',
 'Batas minimal umur tanaman untuk memenuhi kriteria tebu masak adalah 11 bulan.',
 'Piezometer digunakan untuk mengukur kedalaman permukaan air tanah.',
 'Penyakit Pembuluh pada tebu disebabkan oleh bakteri Leaf sonia xyli subsp.xyli.',
 'Benih tebu yang digunakan harus berasal dari varietas tebu unggul yang berasal dari kebun sumber benih yang telah disertifikasi.',
 'B

In [43]:
from rouge_score import rouge_scorer
import numpy as np

def evaluate_rouge_l(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []

    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        scores.append(score["rougeL"].fmeasure)

    return {
        "rougeL_mean": np.mean(scores),
        "rougeL_scores": scores
    }

rouge_result = evaluate_rouge_l(predictions, references)
print("ROUGE-L Mean:", rouge_result["rougeL_mean"])


ROUGE-L Mean: 0.3049464922556807


In [44]:
from bert_score import score

def evaluate_bertscore(predictions, references):
    P, R, F1 = score(
        predictions,
        references,
        lang="id",
        model_type="bert-base-multilingual-cased",
        verbose=True
    )

    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }

bertscore_result = evaluate_bertscore(predictions, references)
print("BERTScore:", bertscore_result)


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.71 seconds, 11.69 sentences/sec
BERTScore: {'precision': 0.7301256060600281, 'recall': 0.760417640209198, 'f1': 0.7433675527572632}


In [45]:
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Berapa ukuran lebar atas got keliling dalam Sistem Reynoso"}
)

print(response["result"])


Lebar atas got keliling adalah 70 cm.


In [46]:
def run_qa(chain, question):
    response = chain.invoke({"query": question})
    return response["result"]
import json

with open("test2.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

questions = [item["question"] for item in test_data]
references = [item["answer"] for item in test_data]


In [47]:
from tqdm.auto import tqdm
import json

results = []      # simpan detail per QA
predictions = []  # hanya jawaban (untuk evaluasi)

for q in tqdm(questions, desc="Running QA Inference", unit="question"):
    try:
        pred = run_qa(qa_chain, q)
    except Exception as e:
        print(f"Error on question: {q}")
        print(e)
        pred = ""

    predictions.append(pred)

    results.append({
        "question": q,
        "prediction": pred
    })



Running QA Inference:   0%|          | 0/20 [00:00<?, ?question/s]

In [48]:
output_data = []

for q, pred, ref in zip(questions, predictions, references):
    output_data.append({
        "question": q,
        "ground_truth": ref,
        "prediction": pred
    })

with open("qa_predictions_gemma3:4btest2newallmini.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

In [49]:
from rouge_score import rouge_scorer
import numpy as np

def evaluate_rouge_l(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = []

    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        scores.append(score["rougeL"].fmeasure)

    return {
        "rougeL_mean": np.mean(scores),
        "rougeL_scores": scores
    }

rouge_result = evaluate_rouge_l(predictions, references)
print("ROUGE-L Mean:", rouge_result["rougeL_mean"])


ROUGE-L Mean: 0.18159079035889175


In [50]:
from bert_score import score

def evaluate_bertscore(predictions, references):
    P, R, F1 = score(
        predictions,
        references,
        lang="id",
        model_type="bert-base-multilingual-cased",
        verbose=True
    )

    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }

bertscore_result = evaluate_bertscore(predictions, references)
print("BERTScore:", bertscore_result)


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 3.05 seconds, 6.55 sentences/sec
BERTScore: {'precision': 0.7205798029899597, 'recall': 0.7194987535476685, 'f1': 0.7192167639732361}
